In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Configurar gerenciamento de ruído com o Estimator

<Accordion>
<AccordionItem title="Versões dos pacotes">

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou versões mais recentes.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Existem várias maneiras de gerenciar o ruído, normalmente usando diversas técnicas de mitigação e supressão de erros para evitar erros antes que eles ocorram. Essas técnicas geralmente causam sobrecarga de pré-processamento. Portanto, é importante alcançar um equilíbrio entre aperfeiçoar seus resultados e garantir que seu job seja concluído em um tempo razoável.

O Estimator suporta as seguintes técnicas de gerenciamento de ruído. Veja [Técnicas de mitigação e supressão de erros](error-mitigation-and-suppression-techniques) para uma explicação de cada uma. Veja a seção [Configurações personalizadas de erros](#advanced-error) para instruções sobre como habilitar essas técnicas.

- [desacoplamento dinâmico](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [twirling de Pauli](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Nível de resiliência
O `resilience_level` especifica quanto de resiliência construir contra erros. Níveis mais altos geram resultados mais precisos, ao custo de tempos de processamento mais longos. Os níveis de resiliência podem ser usados para configurar o trade-off custo/precisão ao aplicar gerenciamento de ruído à sua consulta de primitiva. O gerenciamento de ruído reduz erros (viés) nos resultados processando as saídas de uma coleção, ou conjunto, de circuitos relacionados. O grau de redução de erros depende do método aplicado. O nível de resiliência abstrai a escolha detalhada do método de gerenciamento de ruído para permitir que os usuários raciocinem sobre o trade-off custo/precisão que é apropriado para sua aplicação.

Dado isso, cada nível corresponde a um método ou métodos com nível crescente de sobrecarga de amostragem quântica para permitir que você experimente diferentes trade-offs de tempo-precisão. A tabela a seguir mostra quais níveis e métodos correspondentes estão disponíveis para cada uma das primitivas.

<span id="resilience-table"></span>

| Nível de Resiliência | Descrição                                                                                            | Técnica                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Sem mitigação                                                                                         | Nenhuma                                                                      |
| 1 [Padrão]      | Custos mínimos de mitigação: Mitiga erros associados a erros de leitura               | Twirling de medição [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex)             |
| 2                | Custos médios de mitigação. Normalmente reduz o viés nos estimadores, mas não é garantido que seja sem viés. | Nível 1 + [Extrapolação de Ruído Zero (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) e twirling de gate

> **Info:** Os níveis de resiliência estão atualmente em beta, portanto a sobrecarga de amostragem e a qualidade da solução variarão de circuito para circuito. Novos recursos, opções avançadas e ferramentas de gerenciamento serão lançados de forma contínua. Métodos específicos de gerenciamento de ruído não têm garantia de serem aplicados em cada nível de resiliência.

### Configurar o Estimator com níveis de resiliência
Você pode usar os níveis de resiliência para especificar técnicas de gerenciamento de ruído, ou pode definir técnicas personalizadas individualmente conforme descrito em [Configurações personalizadas de erros](#advanced-error).

> **Note:** Quaisquer opções que você especifique manualmente além do nível de resiliência são aplicadas em adição ao conjunto base de opções definidas pelo nível de resiliência. Portanto, em princípio, você poderia definir o nível de resiliência como 1, mas então desativar a mitigação de medição, embora isso não seja aconselhável.
> 
> Por exemplo, definir o nível de resiliência como 0 desativa `zne_mitigation`, mas `estimator.options.resilience.zne_mitigation = True` substitui esse valor.

### Exemplo
O código a seguir habilita ZNE, TREX e twirling de gate definindo `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Configurações personalizadas de gerenciamento de ruído
Você pode ativar e desativar métodos individuais de gerenciamento de ruído usando as [opções do Estimator](/guides/estimator-options).

> **Note:** Nem todas as opções funcionam juntas em todos os tipos de circuitos. Veja a [tabela de compatibilidade de recursos](estimator-options#options-compatibility-table) para detalhes.

### Exemplo

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: "
    f"{estimator.options.twirling.enable_gates}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>